### Load Dataset

This notebook loads the compact COVID-19 dataset into pandas and focuses on the required **Data Exploration and Cleaning** workflow: initial structure review, missing-value checks, duplicate checks, inconsistency checks, cleaning decisions, and validation after cleaning.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Try common project locations so the notebook can run from either the
# DataCleaning folder or the project root.
data_path_candidates = [
    Path("compact.csv"),
    Path("DataCleaning") / "compact.csv",
    Path("dataset") / "compact.csv",
    Path("..") / "dataset" / "compact.csv"
]

data_path = next((path for path in data_path_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("compact.csv was not found in the current folder, dataset/, or ../dataset/.")

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print(f"Initial shape: {df.shape}")

# Exploration

### Overall Data Structure and Data Types

`df.info()` gives the number of rows, number of columns, data types, and non-null counts. This is the first check for understanding the structure of the dataset.

In [ ]:
print(df.info())

The original dataset has 570606 rows and 61 columns. `country`, `date`, `code`, and `continent` are object columns, while most other columns are numeric. The exact structure should be confirmed from the `df.info()` output above.

### Basic Statistical Summary

`df.describe()` summarizes numeric variables and helps identify unusual ranges, extreme values, and columns that may require additional cleaning.

In [ ]:
desc = df.describe()
display(desc)
desc.to_csv("description.csv")

### Missing-Value Check

The missing-value summary below keeps both the count and percentage of missing values. Sorting by missing percentage makes it easier to identify variables with serious data availability problems.

In [ ]:
missing = df.isnull().sum()
missing_ratio = df.isnull().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_ratio_%": missing_ratio.round(2)
}).sort_values("missing_ratio_%", ascending=False)

print("Missing-value summary, sorted by missing ratio:")
display(missing_summary)
missing_summary.to_csv("missing_data.csv")

### High Missing Ratio Columns and Vaccination Exceptions

Columns with more than 50% missing values are identified as high-missing columns. In general, these columns are removed because they are not reliable enough for core analysis. However, three vaccination rate columns are kept as explicit exceptions because they are important for the project's vaccination-related EDA and optional advanced analysis.

In [ ]:
high_missing_threshold = 50
high_missing_cols = missing_summary.loc[
    missing_summary["missing_ratio_%"] > high_missing_threshold
].index.tolist()

vaccination_cols_to_keep = [
    "people_vaccinated_per_hundred",
    "people_fully_vaccinated_per_hundred",
    "total_vaccinations_per_hundred"
]
existing_vaccination_cols_to_keep = [col for col in vaccination_cols_to_keep if col in df.columns]
retained_high_missing_cols = [col for col in existing_vaccination_cols_to_keep if col in high_missing_cols]

print(f"Columns with more than {high_missing_threshold}% missing values:")
print(high_missing_cols)

if high_missing_cols:
    display(missing_summary.loc[high_missing_cols])
else:
    print("No columns have a missing ratio above 50%.")

print("High-missing vaccination columns kept for later analysis:")
print(retained_high_missing_cols)

Most columns with more than 50% missing values are removed before saving the cleaned dataset. The exception is the three vaccination rate fields: `people_vaccinated_per_hundred`, `people_fully_vaccinated_per_hundred`, and `total_vaccinations_per_hundred`. They are kept because vaccination coverage is a key topic in the EDA and visualization sections. The values are not continuous for most countries, so later analysis should treat them carefully and avoid assuming complete daily coverage.

### Duplicate Data Check

In [ ]:
full_duplicate_count = df.duplicated().sum()
print("Number of fully duplicated rows:", full_duplicate_count)

The output above represents the number of complete duplicate rows. A value of 0 means there are no fully duplicated records.

## Further Inconsistency and Anomaly Checks

This section checks common data-quality issues before cleaning: invalid dates, duplicate country-date records, negative numeric values, logically impossible relationships, and invalid rates.

In [ ]:
# Convert date to datetime; invalid dates become NaT.
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    invalid_date_count = df["date"].isna().sum()
    print("Invalid date count after datetime conversion:", invalid_date_count)
else:
    invalid_date_count = None
    print("Column 'date' was not found.")

# Check duplicate records by country and date.
if {"country", "date"}.issubset(df.columns):
    country_date_duplicate_count = df.duplicated(subset=["country", "date"]).sum()
    print("Duplicate country + date records:", country_date_duplicate_count)
else:
    country_date_duplicate_count = None
    print("Columns 'country' and/or 'date' were not found; country + date duplicate check skipped.")

# Check negative values in numeric columns.
numeric_cols = df.select_dtypes(include="number").columns.tolist()
negative_counts = (df[numeric_cols] < 0).sum().sort_values(ascending=False) if numeric_cols else pd.Series(dtype="int64")
negative_counts = negative_counts[negative_counts > 0]
print("Numeric columns with negative values and their counts:")
if not negative_counts.empty:
    display(negative_counts.to_frame("negative_count"))
else:
    print("No negative values found in numeric columns.")

# Check unreasonable relationships.
if {"total_deaths", "total_cases"}.issubset(df.columns):
    total_deaths_gt_cases = (df["total_deaths"] > df["total_cases"]).sum()
    print("Rows where total_deaths > total_cases:", total_deaths_gt_cases)
else:
    print("total_deaths > total_cases check skipped because one or both columns are missing.")

if {"new_deaths", "new_cases"}.issubset(df.columns):
    new_deaths_gt_cases = (df["new_deaths"] > df["new_cases"]).sum()
    print("Rows where new_deaths > new_cases:", new_deaths_gt_cases)
else:
    print("new_deaths > new_cases check skipped because one or both columns are missing.")

if "positive_rate" in df.columns:
    invalid_positive_rate_count = ((df["positive_rate"] < 0) | (df["positive_rate"] > 1)).sum()
    print("Rows where positive_rate is outside [0, 1]:", invalid_positive_rate_count)
else:
    print("positive_rate column not found; range check skipped.")

if "reproduction_rate" in df.columns:
    negative_reproduction_rate_count = (df["reproduction_rate"] < 0).sum()
    print("Rows where reproduction_rate is negative:", negative_reproduction_rate_count)
else:
    print("reproduction_rate column not found; negative-value check skipped.")

# Cleaning

The cleaning steps below keep the original dataset structure as much as possible while addressing missing values, duplicate records, and basic consistency problems required for the course project.

### Sort by Country and Date

Sorting is important before forward filling cumulative variables. For cumulative COVID-19 fields, the previous observation within the same country is usually the best available estimate when an intermediate value is missing.

In [ ]:
if {"country", "date"}.issubset(df.columns):
    df = df.sort_values(["country", "date"]).reset_index(drop=True)
    print("Data sorted by country and date.")
else:
    print("Sorting by country and date skipped because one or both columns are missing.")

### Missing-Value Handling for Core COVID-19 Measures

Cumulative columns are handled differently from daily new-value columns:

- Cumulative fields are forward-filled within each country, then remaining missing values are filled with 0.
- Daily fields are filled with 0 because a missing daily count is treated as no reported new value for this cleaning workflow.

In [ ]:
cumulative_cols = [
    "total_cases",
    "total_deaths",
    "total_cases_per_million",
    "total_deaths_per_million"
]

daily_cols = [
    "new_cases",
    "new_deaths",
    "new_cases_per_million",
    "new_deaths_per_million"
]

existing_cumulative_cols = [col for col in cumulative_cols if col in df.columns]
existing_daily_cols = [col for col in daily_cols if col in df.columns]

if existing_cumulative_cols:
    before_cumulative_missing = df[existing_cumulative_cols].isna().sum()
    if "country" in df.columns:
        df[existing_cumulative_cols] = df.groupby("country", group_keys=False)[existing_cumulative_cols].ffill()
    else:
        print("Column 'country' not found; cumulative forward fill by country skipped.")
    df[existing_cumulative_cols] = df[existing_cumulative_cols].fillna(0)
    after_cumulative_missing = df[existing_cumulative_cols].isna().sum()
    print("Cumulative columns handled with country-level forward fill, then 0 for remaining missing values:")
    display(pd.DataFrame({
        "missing_before": before_cumulative_missing,
        "missing_after": after_cumulative_missing
    }))
else:
    print("No cumulative columns found for missing-value handling.")

if existing_daily_cols:
    before_daily_missing = df[existing_daily_cols].isna().sum()
    df[existing_daily_cols] = df[existing_daily_cols].fillna(0)
    after_daily_missing = df[existing_daily_cols].isna().sum()
    print("Daily new-value columns handled with fillna(0):")
    display(pd.DataFrame({
        "missing_before": before_daily_missing,
        "missing_after": after_daily_missing
    }))
else:
    print("No daily columns found for missing-value handling.")

### Remove Fully Empty Columns

Columns where every value is missing do not provide analytical information, so they are removed from the cleaned dataset.

In [ ]:
all_missing_cols = df.columns[df.isna().all()]

print("Columns with all values missing:")
print(all_missing_cols.tolist())
df = df.drop(columns=all_missing_cols)

### Remove High-Missing Columns Except Selected Vaccination Fields

The columns stored in `high_missing_cols` have more than 50% missing values. Most of them are removed from the cleaned dataset and excluded from later analysis. The selected vaccination rate columns are retained because they support vaccination-related EDA and presentation insights.

In [ ]:
high_missing_cols_to_drop = [
    col for col in high_missing_cols
    if col in df.columns and col not in existing_vaccination_cols_to_keep
]

print("Columns removed because missing ratio is above 50%:")
print(high_missing_cols_to_drop)

print("High-missing columns retained as vaccination indicators:")
print([col for col in existing_vaccination_cols_to_keep if col in df.columns])

df = df.drop(columns=high_missing_cols_to_drop)

### Duplicate Handling

Both full-row duplicates and country-date duplicates are checked. If multiple records exist for the same country and date, only the first record is kept.

In [ ]:
full_duplicate_count_before = df.duplicated().sum()
print("Fully duplicated rows before duplicate handling:", full_duplicate_count_before)

if full_duplicate_count_before > 0:
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    print("Fully duplicated rows were removed.")
else:
    print("No fully duplicated rows found.")

if {"country", "date"}.issubset(df.columns):
    country_date_duplicate_count_before = df.duplicated(subset=["country", "date"]).sum()
    print("Country + date duplicates before duplicate handling:", country_date_duplicate_count_before)
    if country_date_duplicate_count_before > 0:
        df = df.drop_duplicates(subset=["country", "date"], keep="first").reset_index(drop=True)
        print("Country + date duplicates were removed using keep='first'.")
    else:
        print("No country + date duplicates found.")
else:
    print("Country + date duplicate handling skipped because one or both columns are missing.")

## Validation After Cleaning

The final validation checks confirm that the cleaned dataset can be used for later project analysis and presentation.

In [ ]:
print("Cleaned data shape:", df.shape)

print("Top 20 columns by missing values after cleaning:")
post_clean_missing = df.isna().sum().sort_values(ascending=False).head(20)
display(post_clean_missing.to_frame("missing_count"))

post_full_duplicate_count = df.duplicated().sum()
print("Duplicated rows after cleaning:", post_full_duplicate_count)

if {"country", "date"}.issubset(df.columns):
    post_country_date_duplicate_count = df.duplicated(subset=["country", "date"]).sum()
    print("Country + date duplicates after cleaning:", post_country_date_duplicate_count)
else:
    print("Country + date duplicate validation skipped because one or both columns are missing.")

if "date" in df.columns:
    print("Date dtype after cleaning:", df["date"].dtype)
else:
    print("Date dtype validation skipped because 'date' column is missing.")

if {"total_deaths", "total_cases"}.issubset(df.columns):
    post_total_deaths_gt_cases = (df["total_deaths"] > df["total_cases"]).sum()
    print("Rows where total_deaths > total_cases after cleaning:", post_total_deaths_gt_cases)
else:
    print("total_deaths > total_cases validation skipped because one or both columns are missing.")

if "positive_rate" in df.columns:
    post_invalid_positive_rate_count = ((df["positive_rate"] < 0) | (df["positive_rate"] > 1)).sum()
    print("Rows where positive_rate is outside [0, 1] after cleaning:", post_invalid_positive_rate_count)
else:
    print("positive_rate validation skipped because the column is missing.")

## Save Cleaned Dataset

The cleaned dataset is saved as `compact_clean.csv` for later analysis, reporting, and presentation work.

In [ ]:
df.to_csv("compact_clean.csv", index=False)
print("Saved cleaned dataset to compact_clean.csv")

## Split Cleaned Dataset by Country

After saving `compact_clean.csv`, the cleaned dataset is also split into separate CSV files by the `country` column. This step makes country-level analysis easier because each file contains only one country's time-series records. It also supports later presentation or analysis tasks where a single country needs to be inspected without loading the full global dataset.


In [ ]:
import re

country_output_dir = Path("country")
country_output_dir.mkdir(exist_ok=True)


def safe_filename(value):
    name = str(value).strip()
    name = re.sub(r'[<>:"/\\|?*]+', '_', name)
    name = re.sub(r'\s+', '_', name)
    return name if name else "Unknown"


if "country" in df.columns:
    country_file_count = 0
    country_file_examples = []

    for country_name, country_data in df.groupby("country", dropna=False):
        output_file = country_output_dir / f"{safe_filename(country_name)}.csv"
        if "date" in country_data.columns:
            country_data = country_data.sort_values("date")
        country_data.to_csv(output_file, index=False)
        country_file_count += 1
        if len(country_file_examples) < 10:
            country_file_examples.append(output_file.name)

    print(f"Saved {country_file_count} country-level CSV files to: {country_output_dir}")
    print("Example files:")
    print(country_file_examples)
else:
    print("Column 'country' was not found; country-level split skipped.")
